### Forecasting and Inventory Alert
#### Objective:
##### Generate rolling 7-day sales predictions using trained model and dynamically updated lag features.Compute safety stock, reorder points, and generate restocking alerts based on forecasted demand and service level targets.

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
import joblib

# Load trained model
model = joblib.load(r"outputs/demand_forecast_model.pkl")
features = joblib.load(r"outputs/feature_order.pkl")

import os

folder_path = r"data/processed"
file_name = "extended_history_df.csv"

file_path = os.path.join(folder_path, file_name)



# Load feature data
df = pd.read_csv(r"data/processed/features.csv")
df = df[["date", "store", "item", "sales_log"]].copy()
df["source"] = "actual"
df["date"] = pd.to_datetime(df["date"])

if os.path.exists(folder_path) and file_name in os.listdir(folder_path):
    extended_df = pd.read_csv(file_path)
    extended_df["date"] = pd.to_datetime(extended_df["date"])
    
    # Keep only required columns
    extended_df = extended_df[["date", "store", "item", "sales_log", "source"]].copy()
    
    # Combine actual + extended, but keep latest row per store-item-date
    history_df = pd.concat([df, extended_df], ignore_index=True)
    history_df = (
        history_df
        .sort_values(["store", "item", "date", "source"])
        .drop_duplicates(subset=["store", "item", "date"], keep="last")
        .reset_index(drop=True)
    )
else:
    history_df = df.copy()

# Get latest row per store-item
latest_df = history_df.sort_values("date").groupby(
    ["store", "item"]
).tail(1).reset_index(drop=True)

latest_df.head()
forecasts = []
extended_rows = []

for _, row in latest_df.iterrows():

    store_id = row["store"]
    item_id = row["item"]

    # Get last 14 actual log sales
    history = list(
        history_df[(history_df["store"] == store_id) & (history_df["item"] == item_id)]
        .sort_values("date")["sales_log"]
        .tail(14)
    )

    current_date = row["date"]

    for day in range(1, 8):

        future_date = current_date + pd.Timedelta(days=day)

        lag_1 = history[-1]
        lag_7 = history[-7]
        lag_14 = history[-14]
        roll_mean_7 = np.mean(history[-7:])
        roll_mean_14 = np.mean(history[-14:])
        roll_mean_28 = np.mean(history[-28:])
        roll_std_7 = np.std(history[-7:])


        X = pd.DataFrame([{
            "lag_1": lag_1,
            "lag_7": lag_7,
            "lag_14": lag_14,
            "roll_mean_7": roll_mean_7,
            "roll_mean_14": roll_mean_14,
            "roll_mean_28": roll_mean_28,
            "roll_std_7": roll_std_7,
            "store": store_id,
            "item": item_id,
            "day_of_week": future_date.weekday(),
            "week_of_year": future_date.isocalendar().week,
            "month": future_date.month
        }])
        X = X[features]
        pred_log = model.predict(X)[0]
        pred_sales = np.expm1(pred_log)

        forecasts.append({
            "store": store_id,
            "item": item_id,
            "forecast_date": future_date,
            "predicted_sales": round(pred_sales, 2),
            "predicted_sales_log": pred_log
        })

        # Append predicted log sales to history
        history.append(pred_log)
        extended_rows.append({
            "date": future_date,
            "store": store_id,
            "item": item_id,
            "sales_log": pred_log,
            "source": "forecast"
        })

forecast_df = pd.DataFrame(forecasts)
forecast_df.to_csv(r"outputs/7_day_forecast.csv", index=False)

# Append the forecasted values with history data
extended_history_df = pd.concat(
    [history_df, pd.DataFrame(extended_rows)],
    ignore_index=True
)

# Keep latest unique rows per store-item-date
extended_history_df = (
    extended_history_df
    .sort_values(["store", "item", "date", "source"])
    .drop_duplicates(subset=["store", "item", "date"], keep="last")
    .reset_index(drop=True)
)

extended_history_df.to_csv(r"data/processed/extended_history_df.csv", index=False)


Inventory Alerts

In [ ]:
forecast_df = pd.read_csv(r"outputs/7_day_forecast.csv")
forecast_df["forecast_date"] = pd.to_datetime(forecast_df["forecast_date"])

# Create inventory data
inventory_df = forecast_df[["store", "item"]].drop_duplicates()

# Current inventory is simulated for demonstration purposes
inventory_df["current_inventory"] = np.random.randint(50, 200, size=len(inventory_df))
inventory_df["lead_time_days"] = 7

lead_time_demand = (
    forecast_df
    .groupby(["store", "item"])["predicted_sales"]
    .sum()
    .reset_index(name="lead_time_demand")
)


Z = 1.65  # 95% service level

std_df = df.groupby(
    ["store", "item"]
)["sales"].std().reset_index(name="demand_std")

inventory_df = inventory_df.merge(lead_time_demand, on=["store", "item"])
inventory_df = inventory_df.merge(std_df, on=["store", "item"])

inventory_df["safety_stock"] = (
    Z * inventory_df["demand_std"] * np.sqrt(inventory_df["lead_time_days"])
)


inventory_df["reorder_point"] = (
    inventory_df["lead_time_demand"] + inventory_df["safety_stock"]
)

inventory_df["reorder_alert"] = (
    inventory_df["current_inventory"] < inventory_df["reorder_point"]
)
inventory_df.to_csv(r"outputs/inventory_alerts.csv", index=False)


